In [1]:
import osmnx as ox
import os, random
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import random

In [ ]:
edges_path = r"\data\DSMMetro_edges.csv"
nodes_path = r"\data\DSMMetro_nodes.csv"

In [2]:
def CreateGraph(nodes,edges):
    G = nx.Graph()
    for _, row in nodes.iterrows():
        G.add_node(row['Node'], **row.drop('Node').to_dict())
    for _, row in edges.iterrows():
        G.add_edge(row['u'], row['v'], **row.drop(['u', 'v']).to_dict())
    return G
    
def random_parquet_file(path):
    files = [f for f in os.listdir(path) if f.endswith('.parquet')]
    if not files:
        raise FileNotFoundError("No parquet files found in this directory.")
    return os.path.join(path, random.choice(files))
 

def GetRandomEpisode(filepath=r"C:\Users\asriv\source\repos\Nemesis-Engine\Nemesis-Engine\replays"):
    random_file = random_parquet_file(filepath)
    df = pd.read_parquet(random_file)
    df['timestep'] = df.index
    return df

def EpisodeToReplay(df, max_turns = 119):
    nodes_list = []
    t = []
    color = []
    agentIDs = []
    
    for i,j,k,l in zip(df['seeker_positions'], df['hider_position'], df['timestep'], df['starting_seeker_positions']):
        for seeker, seeker_id in zip(i,l):
            nodes_list.append(seeker)
            color.append('Blue')
            t.append(k)
            agentIDs.append(seeker_id)
        nodes_list.append(j)
        color.append('Red')
        t.append(k)
        agentIDs.append('')
    replay = pd.DataFrame.from_dict({'Agent':agentIDs,
                                    't':t,
                                    'Color':color,
                                    'Node':nodes_list})
    
    replay = replay.merge(nodes, on='Node')
    replay = replay[['Agent', 't', 'Color', 'longitude', 'latitude']]
    max_t = replay['t'].max()
    if max_t < max_turns:
        last_turn = replay[replay['t'] == max_t]
        reds = last_turn[last_turn['Color'] == 'Red'].iloc[0]
        red_idx = last_turn[last_turn['Color'] == 'Red'].index[0]
        blues = last_turn[last_turn['Color'] == 'Blue']
        distances = np.sqrt((blues['latitude'] - reds['latitude'])**2 +
                            (blues['longitude'] - reds['longitude'])**2)
        closest_idx = distances.idxmin()
        closest_blue_agent = blues.loc[distances.idxmin(), 'Agent']
        closest_distance = distances.min()
        chosen_agent = replay.loc[closest_idx, 'Agent']
        green_df = replay[replay['Agent'] == chosen_agent]
        green_df['Color'] = 'Green'

        replay = pd.concat([replay, green_df])
        replay.loc[closest_idx, 'Color'] = 'Green'
        replay.loc[red_idx, 'Color'] = 'Green'
        
    return replay

def AnimateReplay(G, df, i=0, padding=0.25, max_turns = 119):
    green_df = df[df['Color'] == 'Green'].copy(deep=True)
    green_df = green_df.sort_values(by='t')
    df = df[df['Color'] != 'Green'].copy(deep=True) 

    max_t = df['t'].max()
    print(max_t)
    if max_t == 119:
        return go.Figure()

    min_lat, max_lat = df['latitude'].min(), df['latitude'].max()
    min_lon, max_lon = df['longitude'].min(), df['longitude'].max()

    min_lat -= padding
    max_lat += padding
    min_lon -= padding
    max_lon += padding
    sub_nodes = [
                    n for n, data in G.nodes(data=True)
                    if min_lat <= data['latitude'] <= max_lat and min_lon <= data['longitude'] <= max_lon
                ]
    
    G = G.subgraph(sub_nodes).copy()

    # --- 3. Timesteps ---
    timesteps = sorted(df['t'].unique())

    # --- 4. Static roads ---
    lats = nx.get_node_attributes(G, 'latitude')
    lons = nx.get_node_attributes(G, 'longitude')

    fig = go.Figure()

    fig.add_shape(
        type="rect",
        xref="paper", yref="paper",
        x0= 1.03, y0=0.80, x1=1.06, y1=0.8355,  # box height = 0.03
        fillcolor="red",
        line=dict(width=0)
    )
    
    fig.add_annotation(
        x=1.165, y=0.83,                     # y is the vertical center of the box
        xref="paper", yref="paper",
        text="Abductor",
        showarrow=False,
        font=dict(size=14),
        align="left"
    )
    
    fig.add_shape(
        type="rect",
        xref="paper", yref="paper",
        x0= 1.03, y0=0.72, x1=1.06, y1=0.76,  # box height = 0.03 (same as red)
        fillcolor="blue",
        line=dict(width=0)
    )
    
    fig.add_annotation(
        x=1.18, y=0.76,                     # y is vertical center of the box
        xref="paper", yref="paper",
        text="First<br>Responders",
        showarrow=False,
        font=dict(size=14),
        align="left"
    )

    # 🟡 Yellow "Last Seen Area" legend item
    fig.add_shape(
        type="rect",
        xref="paper", yref="paper",
        x0= 1.03, y0=0.64, x1=1.06, y1=0.675,  # same box width, positioned below blue
        fillcolor="rgba(255, 255, 102, 0.8)", # pale yellow fill
        line=dict(width=0)
    )
    
    fig.add_annotation(
        x=1.165, y=0.6575,                    # vertical center of the yellow box
        xref="paper", yref="paper",
        text="Last Seen",
        showarrow=False,
        font=dict(size=14),
        align="left"
    )
    
    fig.add_annotation(
        x=0.97, y=0.25,
        xref="paper", yref="paper",
        text="",
        showarrow=False,
        font=dict(size=14, color="black"),
        align="left",
        name="timestep_label"
    )
    
    for u, v in G.edges():
        fig.add_trace(go.Scattergeo(
            lon=[lons[u], lons[v]],
            lat=[lats[u], lats[v]],
            mode='lines',
            line=dict(width=1, color='gray'),
            showlegend=False,
            hoverinfo='none'
        ))


    agent_lon = df[df['Color'] == 'Red']['longitude'].iloc[0]
    agent_lat = df[df['Color'] == 'Red']['latitude'].iloc[0]
    
    # Circle parameters
    radius = random.uniform(0.08, 0.15)    # degrees (adjust for size)
    lat_offset = random.uniform(-0.1, 0.1)   # up to ~1 km north/south
    lon_offset = random.uniform(-0.05, 0.05)   # up to ~1 km east/west
    
    # Generate circle points
    theta = np.linspace(0, 2*np.pi, 50)
    circle_lon = agent_lon + lon_offset + radius * np.cos(theta)
    circle_lat = agent_lat + lat_offset + radius * np.sin(theta)
    
    # Add circle as a filled polygon
    fig.add_trace(go.Scattergeo(
        lon=circle_lon,
        lat=circle_lat,
        fill="none",
        fillcolor="rgba(255, 255, 102, 0.5)",   # pale yellow fill
        line=dict(color="rgba(255, 255, 102, 0.5)", width=5),
        hoverinfo="skip",
        showlegend=False
    ))
    # --- 5. Frames ---
    frames = []
    for t in timesteps:
        df_up_to_t = df[df['t'] <= t]
        df_current = df[df['t'] == t]

        frame_traces = []

        # 🚨 Build proper RED trails
        red_agents = []
        for agent, track in df_up_to_t[df_up_to_t['Color'] == 'Red'].groupby('Agent'):
            red_agents.append((track['longitude'].tolist(), track['latitude'].tolist()))

        if red_agents:
            red_lon = []
            red_lat = []
            for lon_list, lat_list in red_agents:
                red_lon.extend(lon_list + [None])  # None = break line
                red_lat.extend(lat_list + [None])
            frame_traces.append(go.Scattergeo(
                lon=red_lon,
                lat=red_lat,
                mode='lines',
                line=dict(color='red', width=3),
                hoverinfo='none',
                showlegend=False
            ))


        if (max_t < max_turns and t == max_t):
            green_agents = []
            for agent, track in green_df[green_df['Color'] == 'Green'].groupby('Agent'):

                green_agents.append((track['longitude'].tolist(), track['latitude'].tolist()))
    
            if green_agents:
                green_lon = []
                green_lat = []
                for lon_list, lat_list in green_agents:
                    green_lon.extend(lon_list + [None])  # None = break line
                    green_lat.extend(lat_list + [None])
                frame_traces.append(go.Scattergeo(
                    lon=green_lon,
                    lat=green_lat,
                    mode='lines',
                    line=dict(color='green', width=5),
                    hoverinfo='none',
                    showlegend=False
                ))

            # green_lats = green_df[green_df['Color'] == 'Green']['latitude'].values
            # green_lons = green_df[green_df['Color'] == 'Green']['longitude'].values
            # frame_traces.append(go.Scattergeo(
            #         lon=green_lats,
            #         lat= green_lons,
            #         mode='lines',
            #         line=dict(color='green', width=3),
            #         hoverinfo='none',
            #         showlegend=False
            #     ))

            frame_traces.append(go.Scattergeo(
                lon=[green_df[green_df['Color'] == 'Green']['longitude'].values[-1]],
                lat=[green_df[green_df['Color'] == 'Green']['latitude'].values[-1]],
                mode='markers',
                marker=dict(color='green', size=12),
                name='Green Agents',
                showlegend=False
            ))
            

        # 🚨 Build proper BLUE trails
        blue_agents = []
        for agent, track in df_up_to_t[df_up_to_t['Color'] == 'Blue'].groupby('Agent'):
            if (max_t < max_turns and t == max_t and agent == green_df['Agent'].values[0]):
                continue
            blue_agents.append((track['longitude'].tolist(), track['latitude'].tolist()))
        
        if blue_agents:
            blue_lon = []
            blue_lat = []
            for lon_list, lat_list in blue_agents:
                blue_lon.extend(lon_list + [None])
                blue_lat.extend(lat_list + [None])
            frame_traces.append(go.Scattergeo(
                lon=blue_lon,
                lat=blue_lat,
                mode='lines',
                line=dict(color='blue', width=3),
                hoverinfo='none',
                showlegend=False
            ))

        # --- CURRENT POSITIONS ---
        frame_traces.append(go.Scattergeo(
            lon=df_current[df_current['Color'] == 'Red']['longitude'],
            lat=df_current[df_current['Color'] == 'Red']['latitude'],
            mode='markers',
            marker=dict(color='red', size=9),
            name='Red Agents',
            showlegend=False
        ))

        frame_traces.append(go.Scattergeo(
            lon=df_current[df_current['Color'] == 'Blue']['longitude'],
            lat=df_current[df_current['Color'] == 'Blue']['latitude'],
            mode='markers',
            marker=dict(color='blue', size=9),
            name='Blue Agents'
        ))


        
        static_annotations = [
            dict(x=1.165, y=0.83, xref="paper", yref="paper",
                 text="Abductor", showarrow=False, font=dict(size=14), align="left"),
            dict(x=1.18, y=0.76, xref="paper", yref="paper",
                 text="First<br>Responders", showarrow=False, font=dict(size=14), align="left"),
    dict(
        x=1.165, y=0.6575,                    # vertical center of the yellow box
        xref="paper", yref="paper",
        text="Last Seen",
        showarrow=False,
        font=dict(size=14),
        align="left"
    )
        
        ]
        frames.append(go.Frame(
                        data=frame_traces,
                        name=str(t),
                        layout=go.Layout(
                        annotations=static_annotations + [
                                    dict(
                                        x=1.18, y=0.25,
                                        xref="paper", yref="paper",
                                        text=f"Timestep: {t} min",
                                        showarrow=False,
                                        font=dict(size=14, color="black"),
                                        align="left"
                                    )
                                ]
)
                    ))

    fig.frames = frames

    fig.update_layout(
        width=1200,
        height=1200,
        title=dict(
        text="Modeling Abductions in DSM Metro (Scenario: Static Suspect) ",
        # x=0.475,  # center horizontally
        # y=0.895,
        xanchor='center',
        yanchor='top'
            ),
        #title=,
        geo=dict(
            projection_type='equirectangular',
             fitbounds="locations",              # Ensures it only zooms to your points once
            #lataxis=dict(scaleanchor="lon"),
            lataxis=dict(range=[40.730, 42.470]),
            lonaxis=dict(range=[-95.2, -94.0]),
            showland=True,
            landcolor='rgb(243, 243, 243)',
            subunitwidth=1,
            countrywidth=1,
            subunitcolor='rgb(217, 217, 217)',
            countrycolor='rgb(217, 217, 217)'),
        showlegend=False,
        margin=dict(r=300),
        updatemenus=[dict(
            type='buttons',
            showactive=False,
            x=-0.1, y=0.5,
            buttons=[
                dict(label='▶ Play', method='animate',
                     args=[None, {"frame": {"duration": 1, "redraw": True}, "fromcurrent": True}]),
                dict(label='⏸ Pause', method='animate',
                     args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}])
            ]
        )],
        sliders=[{
            "active": 0,
            "steps": [
                {"args": [[str(t)], {"frame": {"duration": 1, "redraw": True}, "mode": "immediate"}],
                 "label": str(t), "method": "animate"} for t in timesteps
            ]
        }]
    )

    fig.show()
    print("\n\n")
    fig.write_html(f"my_animation{i}.html")

    return fig



def Main(G):
    for i in range(5):
        df = GetRandomEpisode(filepath=r"C:\Users\asriv\source\repos\Nemesis-Engine\Nemesis-Engine\replays")
        replay = EpisodeToReplay(df)
        fig = AnimateReplay(G, replay, i = i)
        fig.data = []
        
    return None





In [3]:
edges = pd.read_csv(edges_path)
nodes = pd.read_csv(nodes_path)
G = CreateGraph(nodes,edges)

In [ ]:
Main(G)